# Module 4: Tune and break

Each section follows a terminal step in the guide. Run a section only after its terminal step.

## 1. Setup

In [ ]:
import json, ssl, urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import labeval as lab  # the evaluation code, next to this notebook (also used by the lab's tests)

NS = lab.namespace()
MONITOR = lab.monitor_url()
rows = lab.load_rows()
print(f"namespace: {NS}\nmonitor:   {MONITOR}")
pd.DataFrame(rows).groupby("label").size().rename("prompts")

In [ ]:
def tick(line):  # one character per prompt: . correct, x wrong, ! no verdict
    outcome, verdict = line.split()[1], line.split()[2]
    print("!" if verdict == "error" else "." if outcome in ("TP", "TN") else "x", end="", flush=True)

def monitor_runs():
    """The live monitor's own tallies (it counted every request you sent through it)."""
    req = urllib.request.Request(MONITOR + "/api/runs", headers={"Authorization": f"Bearer {lab.token()}"})
    ctx = ssl.create_default_context(cafile=f"{lab.SA}/service-ca.crt")
    return {r["run"]: r for r in json.load(urllib.request.urlopen(req, context=ctx, timeout=30))}

def metrics_table(*results):
    return pd.DataFrame([{"run": r["run"], "config": r["config"], "errors as": r["errors_as"],
                          **r["counts"], "errors": r["errors"],
                          **{k: round(v, 2) for k, v in r["metrics"].items()}} for r in results]).set_index("run")

def confusion(ax, res):
    c = res["counts"]
    grid = [[c["TN"], c["FP"]], [c["FN"], c["TP"]]]
    ax.imshow(grid, cmap="Blues")
    for i, row in enumerate(grid):
        for j, v in enumerate(row):
            ax.text(j, i, f"{['TN', 'FP', 'FN', 'TP'][2 * i + j]}\n{v}", ha="center", va="center", fontsize=13,
                    color="white" if v > max(map(max, grid)) / 2 else "black")
    ax.set_xticks([0, 1], ["allowed", "blocked"]); ax.set_yticks([0, 1], ["safe", "unsafe"])
    ax.set_xlabel("guardrails verdict"); ax.set_ylabel("label")
    m = res["metrics"]
    ax.set_title(f"{res['config']}\nP={m['precision']:.2f} R={m['recall']:.2f} FPR={m['fpr']:.2f}")

def wrong(res):
    return pd.DataFrame([r for r in res["results"] if r["outcome"] in ("FP", "FN")],
                        columns=["outcome", "id", "prompt", "rail", "categories"])

## 2. After Exercise 1 (`tuned` = `content-safety` without `S9: PII/Privacy`)

Evaluate `tuned`, compare it with your latest `content-safety` run from Module 3, and look at the
category the model gives "social security number".

In [ ]:
tuned = lab.evaluate("tuned", rows, progress=tick)
print("\n" + lab.summary(tuned))

In [ ]:
runs = monitor_runs()
done = [r for r in runs.values() if r["config"] == "content-safety" and r["n"] == len(rows)]
if done:  # your Module 3 run, as the monitor counted it
    last_cs = max(done, key=lambda r: r["started"])
    cs = {"run": last_cs["run"], "config": "content-safety", "errors_as": last_cs["errors_as"],
          "counts": {k: last_cs[k.lower()] for k in ("TP", "FP", "FN", "TN")}, "errors": last_cs["errors"]}
    cs["metrics"] = lab.metrics(cs["counts"])
else:  # the monitor restarted since Module 3 and forgot it: measure the baseline again
    cs = lab.evaluate("content-safety", rows, progress=tick)
metrics_table(cs, tuned)


In [ ]:
[(r["prompt"], r["verdict"], r["categories"]) for r in tuned["results"] if r["prompt"] == "social security number"]

## 3. After Exercise 2 (`tuned` points at a safety endpoint that does not exist)

The same ten prompts (every fifth: four unsafe, six safe), first as a client that fails open
(an answer without a verdict counts as allowed), then as one that fails closed.

In [ ]:
ten = lab.load_rows(stride=5)
fail_open = lab.evaluate("tuned", ten, errors_as="allowed", progress=tick)
print("\n" + lab.summary(fail_open))
fail_closed = lab.evaluate("tuned", ten, errors_as="blocked", progress=tick)
print("\n" + lab.summary(fail_closed))

In [ ]:
table = metrics_table(fail_open, fail_closed)
cost = table[["fnr", "fpr"]].set_axis(["fails open", "fails closed"])
cost.columns = ["unsafe let through (FNR)", "legitimate refused (FPR)"]
ax = cost.plot.bar(rot=0, ylim=(0, 1.05), figsize=(6, 3.5), color=["#d62728", "#1f77b4"])
ax.set_title("Safety model unreachable: who pays?")
table